# Fine-tune Afro-XLMR (Chichewa school intents)

Works on **Google Colab** and **Kaggle**.

**Colab:** Runtime → Change runtime type → T4 GPU. Upload this project folder (or a zip).

**Kaggle:** Settings → Accelerator GPU, Internet on. Import this notebook and upload the project files.

Run every cell in order. Training the base model takes about 10–20 minutes on a free GPU.

In [ ]:
import os, shutil, sys
from pathlib import Path

IN_KAGGLE = Path("/kaggle").exists()
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

print("Colab:" , IN_COLAB, " | Kaggle:", IN_KAGGLE)

# PyTorch is already on Colab/Kaggle. Only install what training needs.
%pip install -q -U transformers datasets scikit-learn accelerate

## Find the project folder

The cell below looks for `data.json` next to `train.py`. If it cannot find them, upload the project and run it again.

Colab zip example (uncomment if you uploaded a zip):

```python
# !unzip -q /content/text-classfication-model.zip -d /content
```

In [ ]:
roots = [Path.cwd(), Path("/content"), Path("/kaggle/working"), Path("/kaggle/input")]
found = None
for root in roots:
    if not root.exists():
        continue
    for data in root.rglob("data.json"):
        if (data.parent / "train.py").exists():
            found = data.parent
            break
    if found:
        break

if found is None:
    raise SystemExit("Upload data.json, clean.py, train.py, predict.py and run this cell again.")

# Kaggle input is read-only. Copy scripts + data to /kaggle/working.
if IN_KAGGLE and str(found).startswith("/kaggle/input"):
    dest = Path("/kaggle/working/project")
    dest.mkdir(exist_ok=True)
    for name in ["data.json", "clean.py", "train.py", "predict.py"]:
        shutil.copy2(found / name, dest / name)
    found = dest

os.chdir(found)
print("Working in:", Path.cwd())
print("Files:", sorted(p.name for p in Path('.').iterdir() if p.is_file()))

## Clean the dataset

Drops empty rows and near-duplicates (same sentence after ignoring case and punctuation), then groups by label.

In [ ]:
from pathlib import Path
p = Path("data.json")
text = p.read_text()
text = text.replace('"label":"admissio  ns"', '"label":"admissions"')
p.write_text(text)
!python clean.py --input data.json --near-dup

## Train

`train.py` splits into train/val/test (80/10/10, stratified), then fine-tunes Afro-XLMR.

- Out of memory → change `--batch-size 8`
- Faster first run → `--model-name Davlan/afro-xlmr-mini --epochs 3`

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

!python train.py \
  --input data.json \
  --model-name Davlan/afro-xlmr-base \
  --output-dir model_output \
  --epochs 5 \
  --batch-size 16 \
  --learning-rate 2e-5

## Test set (use once)

This file was **not** used to pick the checkpoint. Read the **macro avg** F1, then the confusion matrix to see which labels get mixed up.

In [ ]:
from pathlib import Path
p = Path("predict.py")
p.write_text(p.read_text().replace(
    'y_pred = [r["label"] for r in clf(model_dir)(texts, batch_size=32)]',
    'y_pred = [max(r, key=lambda x: x["score"])["label"] if isinstance(r, list) else r["label"] for r in clf(model_dir, top_k=1)(texts, batch_size=32)]',
))
!python predict.py --model-dir model_output --test-file data/test.json

## Try three messages

You should see `fees`, `password_reset`, then `timetable`.

In [ ]:
!python predict.py --model-dir model_output --text "Kodi ndalama za sukulu ndi zingati?"
!python predict.py --model-dir model_output --text "Ndayiwala password yanga."
!python predict.py --model-dir model_output --text "Kodi timetable ya term yatsopano yatuluka?"

## Save the model

Colab: copies to Google Drive if Drive is mounted, and also zips for download.

Kaggle: Output is already under `/kaggle/working` — Save Version to keep it.

In [ ]:
from pathlib import Path
import shutil
from google.colab import drive, files

assert Path("model_output").exists(), "Train first. model_output is missing."

# 1) Google Drive — stays after Colab closes
drive.mount("/content/drive")
dest = Path("/content/drive/MyDrive/chichewa-school-intent-classifier/model_output")
if dest.exists():
    shutil.rmtree(dest)
shutil.copytree("model_output", dest)
print("On Drive:", dest)

# 2) Zip + force download to your computer
zip_path = shutil.make_archive("/content/model_output", "zip", "model_output")
print("Zip:", zip_path)